# TensorRT benchmark for drive-perception

This machine has no NVIDIA GPU, so the TensorRT numbers come from a Colab GPU runtime
and get merged back into the results table.

Ultralytics builds and runs the TensorRT engine here, not the raw TensorRT API. It tracks
whatever TensorRT version the Colab runtime ships, and that version changes often, so the
notebook keeps running when the API shifts underneath it.

Each model is exported to a TensorRT engine at FP32 and again at FP16, then timed the same
way as the local backends: end to end over predict, with preprocessing and NMS included.
That keeps the numbers comparable to the CPU, MPS and CoreML figures already measured.

## Before you start
1. Runtime, Change runtime type, set the accelerator to a GPU (a T4 is plenty).
2. Run the cells top to bottom.
3. When prompted, upload `yolo11n_kitti.pt` and `yolo11s_kitti.pt` from `models/`.
4. The last cell downloads `tensorrt_results.json`. Put it in the repo under `reports/`.

In [ ]:
# Confirm a GPU is actually attached. If this errors, the runtime has no accelerator.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Ultralytics pulls in a matching TensorRT itself when the engine export runs, so nothing
# else needs pinning here.
!pip install -q ultralytics

from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# Upload the two fine-tuned checkpoints from the repo's models/ directory.
from google.colab import files

uploaded = files.upload()
pt_files = sorted(name for name in uploaded if name.endswith('.pt'))
print('received:', pt_files)
assert pt_files, 'no .pt files uploaded'

In [ ]:
import time
from statistics import median

import numpy as np
from ultralytics import YOLO

# The models were exported for KITTI's wide frames, so the engine input is 224 by 640
# rather than a square. Timing uses random frames of the real KITTI size; content does
# not affect latency, and this keeps the notebook free of any dataset upload.
IMGSZ = [224, 640]
FRAMES = [np.random.randint(0, 255, (375, 1242, 3), dtype=np.uint8) for _ in range(60)]


def time_model(model, warmup=10):
    """End-to-end latency over predict, summarised by the median."""
    for _ in range(warmup):
        model.predict(FRAMES[0], imgsz=IMGSZ, device=0, verbose=False)
    latencies = []
    for frame in FRAMES:
        start = time.perf_counter()
        model.predict(frame, imgsz=IMGSZ, device=0, verbose=False)
        latencies.append((time.perf_counter() - start) * 1000.0)
    latencies.sort()
    med = median(latencies)
    p90 = latencies[min(len(latencies) - 1, round(0.9 * (len(latencies) - 1)))]
    return {
        'median_ms': round(med, 3),
        'p90_ms': round(p90, 3),
        'fps': round(1000.0 / med, 1) if med else 0.0,
    }


print('helper ready')

In [ ]:
# Export a TensorRT engine at each precision, then time it. The engine build takes a
# minute or two per precision, most of it TensorRT searching for the fastest kernels.
gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
results = {'gpu': gpu_name[0].strip(), 'imgsz': IMGSZ, 'models': {}}

for pt in pt_files:
    model_name = pt.replace('.pt', '')
    results['models'][model_name] = {}
    for precision, half in [('fp32', False), ('fp16', True)]:
        print(f'building {model_name} {precision} engine ...')
        engine_path = YOLO(pt).export(
            format='engine', imgsz=IMGSZ, half=half, device=0, verbose=False
        )
        stats = time_model(YOLO(engine_path, task='detect'))
        results['models'][model_name][precision] = stats
        print(f'  {precision}: {stats["median_ms"]} ms median, {stats["fps"]} FPS')

print('\ndone')

In [ ]:
# Readable table plus the FP16 speedup over FP32.
print(f"GPU: {results['gpu']}\n")
print(f"{'model':16s} {'precision':10s} {'median ms':>10s} {'p90 ms':>8s} {'FPS':>8s}")
for model, prec in results['models'].items():
    for name, stats in prec.items():
        print(f"{model:16s} {name:10s} {stats['median_ms']:>10.3f} {stats['p90_ms']:>8.3f} {stats['fps']:>8.1f}")
    if 'fp32' in prec and 'fp16' in prec:
        speedup = prec['fp32']['median_ms'] / prec['fp16']['median_ms']
        print(f"  -> FP16 is {speedup:.2f}x faster than FP32 for {model}\n")

In [ ]:
# Save and download. Drop this file into the repo under reports/.
import json

with open('tensorrt_results.json', 'w') as f:
    json.dump(results, f, indent=2)

from google.colab import files

files.download('tensorrt_results.json')


INT8 is the next step. It needs a small calibration set, and once measured it shows what
dropping to 8-bit costs in accuracy for the extra speed.